In [1]:
import os
import arcpy

In [13]:
#lists directories where output from arcgis were stored
source_dir_list = [r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\Initial Catchments\wall', r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\Initial Catchments\wall\0,5']

In [14]:
#lists destination directory for catchments after each step 
#variable naming = function name + destination
valid_name_cat_dest = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\valid_name_catchment' #destination for catchment polygons
valid_name_snap_dest = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\valid_name_snappoint' #destination for catchment snappoints
simplify_catchment_dest = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\simplified_catchment'
smoothen_catchment_dest = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\smoothened_catchment'
kanal_erased_dest = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\kanal_erased_catchment'
# kanal_erase

In [1]:
def valid_name(dir_list, cat_dest, snap_dest,merged_pegelID = r'C:\Users\Adhikari\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\Runoff_gauge_UTM32_Merge'):
    '''
    a,The watersheds were named as PegelID_GaugeName.shp(e.g. 12_Watershed.shp), and snappoints as PegelID_GaugeName_Snappoint.shp . This violated the rule that name of shapefile 
    within a geodatabase couldnot start from a number or special character. So, renaming them was required. Likewise, different directories were used to store 0,5 
    and 1 usage watershed and snappoint files. During this step, all polygon shapefiles are copied to a directory, and all point shapefiles to another directory. 
    
    b,The attributes required but not joined from Gauge ID information are also joined. The table is updated only in target directory. 
    
    Function arguments: 
    dir_list = list of directories where the required shapefiles with erroneous names are. .
    cat_dest = destination directory for catchments. All catchments, of both usage class, are saved in this directory.
    snap_dest = destination directory for snappoint. All snappoint, of both usage class, are saved in this directory.
    merged_pegelID = file path of shapefile containing gauges of both 0,5 and 1 usage. '''
    arcpy.env.overwriteOutput = True
    j = 1 #counter to show total number of shapefiles dealt with
    for source_dir in dir_list:
        arcpy.env.workspace = source_dir
        feature_classes = arcpy.ListFeatureClasses()
        for fc in feature_classes:
            source_path = os.path.join(source_dir, fc)
            # arcpy.management.CalculateGeometryAttributes('source_path', ['Area_sq_km', 'AREA'], 'SQUARE_KILOMETERS')
            new_valid_name = 'i_' + (fc.replace('-','')).replace(' ', '')[:7]
            try:
                if 'Smappoint' in fc or 'Snappoint' in fc: #points # Smappoint because one snappoint was erronously named smappoint
                    target_path = os.path.join(snap_dest,new_valid_name)
                else: #polygons 
                    target_path = os.path.join(cat_dest,new_valid_name)
            except PermissionError: #this part bypasses lockfiles used in jupyter or arc
                print(f'Ignoring {source_path} as lockfile')
                pass
            arcpy.CopyFeatures_management(source_path, target_path)
            arcpy.JoinField_management(target_path,'Pegel_ID',merged_pegelID, 'Pegel_ID', ['Usage', 'Maintainan'])
            j += 1 
    print('Done',j)

In [46]:
valid_name(dir_list = source_dir_list, cat_dest = valid_name_cat_dest, snap_dest = valid_name_snap_dest)

Done 216


In [47]:
def select_fields(dir_fld):
    '''
    a, Only required fields are selected from all the shapefiles: all the polygon shapefiles will have the same 
    fields, and all point shapefiles another same list of fields. This is required in order to maintain compatibility in the same feature class.

    Input arguments: 
    dir_fld = shapefile attribute dicitonary with structure as: { shp_type:{'load_dir' : source directory, 'required_fields': list of required fields} }

    The \texttt{shp\_type} is required only to maintain distinction between catchments (polygons) and snappoints (points). Later, when only polygons are required, a copy of this dictionary without the points can be made. Indexing this dictionary as dict[polygon] will result in error. 
    
    Example of the dictionary:
    shp_dictionary = {'catchment':{'load_dir': valid_name_cat_dest, 'required_fields' : ['FID','Shape','Pegel_ID','Gauge_Name','River_Name','Area_PVZ_f','Area_sq_km','SnapDist','Usage','Maintainan']}, 
        'snappoints': {'load_dir': valid_name_snap_dest, 'required_fields' : ['OID','FID','Shape','Pegel_ID','Gauge_Name','River_Name','SnapDist','Usage','Maintainan']}} 
    '''
    for key in dir_fld.keys(): #iterates over each shapefile type
        #key_name = key #getting key as a variable for indexing the dictionary
        arcpy.env.workspace = dir_fld[key]['load_dir']
        feature_classes = arcpy.ListFeatureClasses()
        for fc in feature_classes:
            shp_source_path = os.path.join(dir_fld[key]['load_dir'], fc)
            shp_fields = arcpy.ListFields(shp_source_path)
            fields_to_delete = [field.name for field in shp_fields if field.name not in dir_fld[key]['required_fields']]
            # print(fields_to_delete)
            for field in fields_to_delete:
                try:
                    arcpy.DeleteField_management(shp_source_path, field)
                except arcpy.ExecuteError as e: #if some field like OID cannot be deleted, it raises an error and continues the operation on other fields
                    print(f"Error deleting field {field}: {e}")
    print('Done')

In [48]:
shp_dictionary = {'catchment':{'load_dir': valid_name_cat_dest, 'required_fields' : ['FID','Shape','Pegel_ID','Gauge_Name','River_Name','Area_PVZ_f','Area_sq_km','SnapDist','Usage','Maintainan']}, 
        'snappoints': {'load_dir': valid_name_snap_dest, 'required_fields' : ['OID','FID','Shape','Pegel_ID','Gauge_Name','River_Name','SnapDist','Usage','Maintainan']}}
# dictionary structure = {shp_type:{source directory, list of required fields}}

select_fields(dir_fld = shp_dictionary)

Done


## Skip; 
This part simplifies and smoothens catchments individually.
This violated topological rules. 

In [14]:
def simplify_catchment(input_dir, output_dir):
    arcpy.env.workspace = input_dir
    arcpy.env.overwriteOutput = True
    input_shps = arcpy.ListFeatureClasses()
    i = 0
    for shp in input_shps:
        outpath = os.path.join(output_dir,shp)
        arcpy.cartography.SimplifyPolygon(shp, outpath, 
                                  "POINT_REMOVE", 20, 400, "RESOLVE_ERRORS", 
                                  "NO_KEEP")
        i += 1
    print('Done',i)

In [15]:
simplify_catchment(input_dir = valid_name_cat_dest, output_dir = simplify_catchment_dest)

Done 107


In [19]:
def smoothen_catchment(input_dir, output_dir):
    arcpy.env.workspace = input_dir
    arcpy.env.overwriteOutput = True
    input_shps = arcpy.ListFeatureClasses()
    i = 0
    for shp in input_shps:
        outpath = os.path.join(output_dir,shp)
        arcpy.cartography.SmoothPolygon(shp, outpath, "PAEK", 60, "FIXED_ENDPOINT", 
                 "RESOLVE_ERRORS")
        i += 1
    print('Done',i)

In [20]:
smoothen_catchment(input_dir = simplify_catchment_dest, output_dir = smoothen_catchment_dest)

Done 107


In [29]:
def kanal_erase(input_dir,output_dir, kanal_dir = r'C:\Users\Adhikari\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\DrainBasin_Waterbody_LOCAL_kanal_to_erase'):
    arcpy.env.workspace = input_dir
    arcpy.env.overwriteOutput = True
    input_shps = arcpy.ListFeatureClasses()
    i = 0
    for shp in input_shps:
        outpath = os.path.join(output_dir,shp)
        arcpy.Erase_analysis(shp, kanal_dir, outpath)
        i += 1
    print('Done',i)
                     

In [30]:
kanal_erase(input_dir=smoothen_catchment_dest,output_dir=kanal_erased_dest)

Done 107


In [31]:
shp_dictionary_final = {'catchment':{'load_dir': kanal_erased_dest, 'required_fields' : ['FID','Shape','Pegel_ID','Gauge_Name','River_Name','Area_PVZ_f','Area_sq_km','SnapDist','Usage','Maintainan']}}
select_fields(dir_fld = shp_dictionary_final)

Done


In [50]:
write_to_featureclass(shp_dir = valid_name_cat_dest, fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\input')

Removed old files
All shapefiles appended successfully.


## Continue
Here the catchments are compiled into a single gdb and simplification and smoothening are done such that topological relations are maintained.

In [51]:
def write_to_featureclass(shp_dir, fc_path):
    '''
    a, Writes all shapefiles into a feature class. This does not create a geodatabase, and feature dataset. They were done manually.
    This part requires that all required polygon shapefiles are in one directory, and point shape files are in another directory.
    The valid_name() function ensures this in this script. 
    Input arguments:
    shp_dir = directory containing all shapefile of a type
    fc_path = path to the feature class'''
    arcpy.env.workspace = shp_dir
    arcpy.env.overwriteOutput = True
    arcpy.DeleteRows_management(fc_path) #removes rows (records) added to the featureclass from previous iteration
    print('Removed old files')
    shapefiles = arcpy.ListFeatureClasses("*.shp")
    for shapefile in shapefiles:
        try:
            arcpy.Append_management(shapefile, fc_path, "NO_TEST")
            #print(f"Appended {shapefile} to {appended_fc}")
        except arcpy.ExecuteError as e:
            print(f"Error appending {shapefile}: {e}")
    print("All shapefiles appended successfully.")

In [52]:
def simplify_smoothen_erase_kanal(gdb_dir,input_fc, simplified_fc, smoothened_fc, kanal_erased_fc, simplification_dist , smoothening_dist, erase_kanal_fc):
    '''
    a, This functions takes the feature class containing all the polygon delineated from ArcGIS as input.  
    b, Then, it simplifies them using DouglasPeuker algorithm, smoothens them using PAEK, and erases kanals from the waterbodies.
    Algorithm chosen for simplification and smoothening are not function arguments, as different algorithms require different input parameters.
    Working with feature classes ensures that topological rules are maintained between adjacent catchments.  

    Input parameters: 
    gdb_dir = path of  geodatabase containing the feature class of all delineated catchments. All results from this function are saved within this geodatabase. 
    input_fc = path of the feature class. This is the output of write_to_featureclass() for watershed polygons. 
    simplified_fc = path of feature class containing all simplified watershed. If the feature class does not exit, it is created.
    smoothened_fc = path of feature class containing all smoothened watershed. If the feature class does not exit, it is created.
    simplification_dist = threshold distance parameter for simplifying algorithm. More information at arcgis documentation. 
    smoothening_dist = parametric averaging distance for smoothening algorithm. More information at arcgis documentation.
    erase_kanal_fc = path of the shapefile containing canals that have to be erased.

    While rerunning, records in all featureclasses except input fc are erased first.
     '''
    arcpy.env.workspace = gdb_dir
    template = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\valid_name_catchment\i_3611103.shp'
    spatial_ref = arcpy.Describe(template).spatialReference
    geometry_type = "POLYGON"
    has_m = "DISABLED"
    has_z = "DISABLED"

    #Generalization
    write_to_featureclass(shp_dir = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\valid_name_catchment', fc_path = input_fc) 
    #if generalization is done, this modifies the input_fc itself, so the input fc has to be created again 
    # arcpy.edit.Generalize(input_fc, 20) 
    # print('Generalized!')
    
    #Simplification
    if not arcpy.Exists('Simplified'): #if the featureclass doesn't exist, it is created
        arcpy.management.CreateFeatureclass(gdb_dir, 'Simplified', 'POLYGON', template, has_m, has_z, spatial_ref)
        
    arcpy.DeleteRows_management(simplified_fc) #removes rows (records) added to the featureclass from previous iteration
    arcpy.cartography.SimplifyPolygon(input_fc, simplified_fc, 
                                  "POINT_REMOVE", simplification_dist, 400, "RESOLVE_ERRORS", 
                                  "NO_KEEP")
    print('Simplified')
    
    #Smoothening
    if not arcpy.Exists('Smoothened'):
        arcpy.management.CreateFeatureclass(gdb_dir, 'Smoothened', 'POLYGON', template, has_m, has_z, spatial_ref)
        
    arcpy.DeleteRows_management(smoothened_fc)
    arcpy.cartography.SmoothPolygon(simplified_fc, smoothened_fc, "PAEK", smoothening_dist, "FIXED_ENDPOINT", 
                 "RESOLVE_ERRORS")
    print('Smoothened')

    if not arcpy.Exists(kanal_erased_fc):
        arcpy.management.CreateFeatureclass(gdb_dir, 'Kanal_erased', 'POLYGON', template, has_m, has_z, spatial_ref)
    arcpy.analysis.Erase(smoothened_fc, erase_kanal_fc, kanal_erased_fc) #performs geprocessing operation Erase
    print('Kanals Erased')
    

In [ ]:
write_to_featureclass(shp_dir = valid_name_cat_dest, fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\input')

Removed old files
All shapefiles appended successfully.


In [53]:
gdb_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints'
input_fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\input'
simplified_fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\Simplified'
smoothened_fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\Smoothened'
erased_fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\Kanal_erased'
kanal_to_erase_path = r'C:\Users\Adhikari\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\DrainBasin_Waterbody_LOCAL_kanal_to_erase'
simplify_smoothen_erase_kanal(gdb_dir = gdb_path,input_fc = input_fc_path, simplified_fc=simplified_fc_path, smoothened_fc=smoothened_fc_path, kanal_erased_fc = erased_fc_path, simplification_dist = 20, smoothening_dist = 80, erase_kanal_fc = kanal_to_erase_path)

Removed old files
All shapefiles appended successfully.
Simplified
Smoothened
Kanals Erased


In [ ]:
def finalize(gdb_path, final_fc):
    '''
    a, This function creates the final output. It keeps only the required fields, calculates area of the catchment after simplify_smoothen_erase_kanal(), and 
    calculates percentage difference with literature.
    Input parameters:
    gdb_path = path to the geodatabase containing the final feature class
    final_fc = name of the final feature class'''
    arcpy.env.workspace = gdb_path
    final_fc_path = os.path.join(gdb_path,final_fc)
    
    #Keep only required attributes to the final output
    shp_dictionary = {'catchment':{'load_dir': erased_fc_path, 'required_fields' : ['FID','Shape','Pegel_ID','Gauge_Name','River_Name','Area_PVZ_f','Area_sq_km','SnapDist','Usage','Maintainan']}, 
        'snappoints': {'load_dir': valid_name_snap_dest, 'required_fields' : ['OID','FID','Shape','Pegel_ID','Gauge_Name','River_Name','SnapDist','Usage','Maintainan']}}
    fields_to_delete = [field.name for field in arcpy.ListFields(final_fc) if field.name not in shp_dictionary['catchment']['required_fields']]
    for field in fields_to_delete:
    try:
        arcpy.DeleteField_management(final_fc_path, field)
    except arcpy.ExecuteError as e:
        print(f"Error deleting field {field}: {e}")

    #Calculates the area of watershed after removing canals
    arcpy.AddField_management(final_fc, "Shape_area", "DOUBLE")
    exp = "!SHAPE.AREA@SQUAREKILOMETERS!"
    arcpy.CalculateField_management(final_fc, "Area_smooth", exp, "PYTHON_9.3")

    #Calculates percentage area difference with respect to literature
    arcpy.management.AddField(final_fc, 'Final_Area_diff', 'FLOAT', 4)
    expression = 'getAreaDiff(!Area_PVZ_f!,!Area_sq_km!)'  #name of fields
    codeblock = '''def getAreaDiff (a_ref, a_fin):
        return ((float(a_ref) - float(a_fin))/float(a_ref) * 100)
        '''
    arcpy.management.CalculateField(final_fc, 'Final_Area_diff', expression, "PYTHON3", codeblock )

In [57]:
#Keep only required attributes to the final output
arcpy.env.workspace = gdb_path
[field.name for field in arcpy.ListFields('Kanal_erased')]
shp_dictionary = {'catchment':{'load_dir': erased_fc_path, 'required_fields' : ['FID','Shape','Pegel_ID','Gauge_Name','River_Name','Area_PVZ_f','Area_sq_km','SnapDist','Usage','Maintainan']}, 
        'snappoints': {'load_dir': valid_name_snap_dest, 'required_fields' : ['OID','FID','Shape','Pegel_ID','Gauge_Name','River_Name','SnapDist','Usage','Maintainan']}}
# dictionary structure = {shp_type:{source directory, list of required fields}}

fields_to_delete = [field.name for field in arcpy.ListFields('Kanal_erased') if field.name not in shp_dictionary['catchment']['required_fields']]
# print(fields_to_delete)
for field in fields_to_delete:
    try:
        arcpy.DeleteField_management(erased_fc_path, field)
    except arcpy.ExecuteError as e:
        print(f"Error deleting field {field}: {e}")

#Calculates the area of watershed after removing canals
arcpy.AddField_management('Kanal_erased', "Shape_area", "DOUBLE")
exp = "!SHAPE.AREA@SQUAREKILOMETERS!"
arcpy.CalculateField_management('Kanal_erased', "Area_smooth", exp, "PYTHON_9.3")

#Calculates percentage area difference with respect to literature
arcpy.management.AddField('Kanal_erased', 'Final_Area_diff', 'FLOAT', 4)
#arcpy.management.CalculateGeometryAttributes('Kanal_erased', ['Area_after_smoothening', 'AREA'], '#', 'SQUARE_KILOMETERS')
expression = 'getAreaDiff(!Area_PVZ_f!,!Area_sq_km!)'
codeblock = '''def getAreaDiff (a_ref, a_fin):
    return ((float(a_ref) - float(a_fin))/float(a_ref) * 100)
    '''
arcpy.management.CalculateField('Kanal_erased', 'Final_Area_diff', expression, "PYTHON3", codeblock )

Error deleting field OBJECTID: Failed to execute. Parameters are not valid.
ERROR 001334: Cannot delete required field OBJECTID
Failed to execute (DeleteField).

Error deleting field Shape_Length: Failed to execute. Parameters are not valid.
ERROR 001334: Cannot delete required field Shape_Length
Failed to execute (DeleteField).

Error deleting field Shape_Area: Failed to execute. Parameters are not valid.
ERROR 001334: Cannot delete required field Shape_Area
Failed to execute (DeleteField).



<Result 'C:\\Users\\Adhikari\\Desktop\\Catchment Delineation\\Work\\python_outputs\\final.gdb\\catchment_and_snappoints\\Kanal_erased'>

In [40]:
valid_name(dir_list = source_dir_list, cat_dest = valid_name_cat_dest, snap_dest = valid_name_snap_dest)
select_fields(dir_fld = shp_dictionary)
write_to_featureclass(shp_dir = valid_name_snap_dest, fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\snappoints')
write_to_featureclass(shp_dir = valid_name_cat_dest, fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\input')


simplify_catchment(input_dir = valid_name_cat_dest, output_dir = simplify_catchment_dest)
smoothen_catchment(input_dir = simplify_catchment_dest, output_dir = smoothen_catchment_dest)
kanal_erase(input_dir=smoothen_catchment_dest,output_dir=kanal_erased_dest)
select_fields(dir_fld = shp_dictionary_final)
write_to_featureclass(shp_dir = kanal_erased_dest, fc_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\Work\python_outputs\final.gdb\catchment_and_snappoints\catchments')

Done 216
Done 107
Done 107
Done 107
Done
Removed old files
All shapefiles appended successfully.
